# Combined gold contracts — 7B NF4 repair
**Generator: pinned Qwen2.5-3B-Instruct, unquantized FP16. Repair: pinned Qwen2.5-7B-Instruct, NF4 with float16 computation.** Uses the same gold contracts, combined evidence/recovery loop, schema guard, prompts, seed and budgets as the completed combined 3B gold run (84 correct, zero incorrect accepted, 16 withheld). No intent parser, adapter or training. One model resident at a time; generator on GPU 1, repair on GPU 0 with T4 x2.

1. Upload **stage2_combined_gold_7b_source.zip** as a private Kaggle dataset and attach it. Intact ZIP and automatically extracted source are supported.
2. Enable **Internet**, choose **T4 x2**, then **Save Version → Run All**.
3. The notebook runs **2 smoke tasks → 20 development tasks → 100 evaluation tasks**. The 20-task development stage is followed by a separate full-evaluation cell. Each uses a fresh output directory.
4. Download **stage2_combined_gold_7b.zip**, **stage2_combined_gold_7b_summary.zip** and the executed notebook. Partial outputs are not final results.

No GitHub push or previous results are needed. Optional: attach the completed **stage2_combined_gold_3b.zip** as a separate dataset for the last CPU comparison cell; it is never used during inference. The new collector fixes the numeric YAML mapping-key versus JSON representation issue and retains strict validation. The control's original files remain unchanged.

This changes both repair model size and precision. It is a repair-configuration comparison, not a pure model-size ablation. Previously inspected tasks are diagnostic. Fresh initial generation may differ and the optional comparison reports initial-code agreement. A mismatch in software/GPU environment or other settings stops only comparison; completed inference ZIPs remain available. CPU checks cannot guarantee GPU execution or improved model quality.


In [ ]:
import datetime,hashlib,json,os,shutil,subprocess,sys,zipfile
from pathlib import Path
INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working')
WORK.mkdir(parents=True,exist_ok=True)
RUN_ID=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR=WORK/('run_'+RUN_ID);RUN_DIR.mkdir()
DRIVE_RESULTS=None
def save_results():
    if DRIVE_RESULTS is not None:
        DRIVE_RESULTS.mkdir(parents=True,exist_ok=True)
        for p in RUN_DIR.iterdir():
            if p.is_file() and p.suffix in ['.zip','.log','.json','.txt']:shutil.copy2(p,DRIVE_RESULTS/p.name)
        print('Drive results:',DRIVE_RESULTS)
print('Run folder:',RUN_DIR)



In [ ]:
MANIFEST_SHA='1cd923de600a419d8b6299cafab1fa8afe70f3016b9fddc3e51cf55609e1bcf8'
ROOT=WORK/('stage2_combined_gold_7b_source_'+MANIFEST_SHA[:12])
def unpack(raw,read):
    assert hashlib.sha256(raw).hexdigest()==MANIFEST_SHA,'Wrong source ZIP version'
    manifest=json.loads(raw);assert manifest['experiment']=='stage2-combined-gold-7b-nf4-v1'
    for name,sha in manifest['sha256'].items():
        relative=Path(name);assert not relative.is_absolute() and '..' not in relative.parts
        data=read(name);assert hashlib.sha256(data).hexdigest()==sha,f'Changed source/input: {name}'
        path=ROOT/relative;path.parent.mkdir(parents=True,exist_ok=True)
        if path.exists():assert path.read_bytes()==data,f'Changed working source: {name}'
        else:path.write_bytes(data)
    sidecar=ROOT/'_stage2_combined_gold_7b_manifest.json'
    if sidecar.exists():assert sidecar.read_bytes()==raw,'Changed source manifest'
    else:sidecar.write_bytes(raw)
    print('Verified fresh-run source/data:',len(manifest['sha256']),'files',MANIFEST_SHA)
found=False
for path in INPUT.rglob('manifest.json'):
    if hashlib.sha256(path.read_bytes()).hexdigest()==MANIFEST_SHA:
        unpack(path.read_bytes(),lambda name:(path.parent/name).read_bytes());found=True;break
if not found:
    for path in INPUT.rglob('stage2_combined_gold_7b_source.zip'):
        with zipfile.ZipFile(path) as z:
            assert len(z.namelist())==len(set(z.namelist()))
            prefix='stage2_combined_gold_7b_source/';name=prefix+'manifest.json'
            if name not in z.namelist():continue
            raw=z.read(name)
            if hashlib.sha256(raw).hexdigest()!=MANIFEST_SHA:continue
            unpack(raw,lambda name:z.read(prefix+name));found=True;break
assert found,'Attach/place stage2_combined_gold_7b_source.zip in the input folder.'
PACKAGES=WORK/'stage2_combined_gold_7b_packages'
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(PACKAGES),'-r',str(ROOT/'requirements.txt')],check=True)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[ROOT,ROOT/'src',PACKAGES])),PYTHONUNBUFFERED='1',TOKENIZERS_PARALLELISM='false')


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.20.2','peft==0.21.2','bitsandbytes==0.50.2','accelerate==1.15.0','tokenizers==0.23.3','safetensors==0.8.0','huggingface-hub==1.33.0'],check=True)
subprocess.run([sys.executable,'-m','eval.stage2_independent_v2.environment','--prepare'],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-c',"import torch; assert torch.cuda.is_available(); assert all(torch.cuda.get_device_properties(i).total_memory/2**30>=12 for i in range(torch.cuda.device_count())); print(torch.__version__,[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-m','finetune.selective_facts.attention'],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-m','eval.stage2_combined_gold_7b.preflight'],cwd=ROOT,env=ENV,check=True)


In [ ]:
DEV_COUNT=20
def execute(smoke,split='test'):
    name='stage2_combined_gold_7b'+('_smoke' if smoke else '_dev' if split=='dev' else '')
    out=RUN_DIR/name;assert not out.exists(),'Use a fresh run folder.'
    command=[sys.executable,'-m','eval.stage2_combined_gold_7b.runner','--method','gold','--preset','gold_contract','--out',str(out),'--packages',str(PACKAGES),'--split',split]
    if smoke:command+=['--smoke']
    log=RUN_DIR/(name+'.log')
    try:
        with log.open('w') as f:
            process=subprocess.Popen(command,cwd=ROOT,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in process.stdout:
                print(line,end='');f.write(line);f.flush()
            code=process.wait()
    finally:save_results()
    assert code==0,f'Run failed: inspect {log} and the partial ZIP. A partial job is not a result.'
    assert json.loads((out/'completion.json').read_text())['complete']
    assert len(list((out/'tasks').glob('*.json')))==(2 if smoke else DEV_COUNT if split=='dev' else 100)
    return out.with_suffix('.zip')
print('STAGE 1/3: two infrastructure smoke tasks')
SMOKE=execute(True)
print('STAGE 2/3: 20 development tasks; the next cell runs 100 test tasks')
DEV=execute(False,split='dev')
print('Development summary:',(DEV.with_suffix('')/'summary.json').read_text())
print('Smoke and end-to-end development completed. Inspect the development summary; the next cell runs 100 diagnostic test tasks freshly.')


In [ ]:
print('STAGE 3/3: full 100-task diagnostic evaluation')
BUNDLE=execute(False)
OUT=RUN_DIR/'stage2_combined_gold_7b_summary'
subprocess.run([sys.executable,'-m','eval.stage2_combined_gold_7b.collect','--gold',str(BUNDLE),'--out',str(OUT)],cwd=ROOT,env=ENV,check=True)
from IPython.display import display,FileLink,Markdown
display(Markdown((OUT/'SUMMARY.md').read_text()))
display(FileLink(str(BUNDLE)));display(FileLink(str(OUT.with_suffix('.zip'))))
print('Download the gold bundle and this executed notebook. The completed 3B gold bundle is optional in the next cell.')


In [ ]:
# Optional CPU comparison with the completed 3B FP16 gold run (84/0/16).
# Attach stage2_combined_gold_3b.zip as a separate dataset. No previous output is used for inference.
candidates=list(INPUT.rglob('stage2_combined_gold_3b.zip'))
for path in INPUT.rglob('plan.json'):
    try:
        plan=json.loads(path.read_text())
        if plan.get('experiment')!='stage2-combined-gold-3b-fp16-v1' or plan.get('smoke') or plan.get('split')!='test':continue
        folder=path.parent;seal=json.loads((folder/'bundle.json').read_text())
    except (OSError,ValueError):continue
    assert seal['complete'],'Incomplete control bundle'
    local=RUN_DIR/('repacked_control_'+str(len(candidates))+'.zip')
    with zipfile.ZipFile(local,'w',zipfile.ZIP_DEFLATED) as archive:
        for name,digest in seal['sha256'].items():
            relative=Path(name);assert not relative.is_absolute() and '..' not in relative.parts
            data=(folder/relative).read_bytes();assert hashlib.sha256(data).hexdigest()==digest
            archive.writestr(name,data)
        archive.writestr('bundle.json',json.dumps(seal))
    candidates.append(local)
distinct={}
for path in candidates:
    with zipfile.ZipFile(path) as z:seal=json.loads(z.read('bundle.json'))
    distinct[json.dumps(seal,sort_keys=True)]=path
assert len(distinct)<=1,'Multiple different complete 3B controls attached'
if distinct:
    OUT=RUN_DIR/'stage2_combined_gold_7b_comparison'
    subprocess.run([sys.executable,'-m','eval.stage2_combined_gold_7b.collect','--gold',str(BUNDLE),'--baseline',str(next(iter(distinct.values()))),'--out',str(OUT)],cwd=ROOT,env=ENV,check=True)
    display(Markdown((OUT/'SUMMARY.md').read_text()));display(FileLink(str(OUT.with_suffix('.zip'))))
else:
    print('No 3B control attached. The 7B run and summary are complete; download them for local comparison.')
